In [0]:
%run ../config/set_up_env_paths

In [0]:
from config.core import config
from pyspark.sql.functions import col , when, count, to_date, round, upper, trim, current_date, length, lit
import pyspark
from pyspark.sql.types import DecimalType

In [0]:
table = dbutils.widgets.get("table") 
rate_type = dbutils.widgets.get("rate_type")
update_frequency = dbutils.widgets.get("update_frequency")

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-7139859878850383>, line 1
----> 1 table = dbutils.widgets.get("table")

File /databricks/python_shell/lib/dbruntime/WidgetHandlerImpl.py:82, in WidgetsHandlerImpl.get(self, name)
     42 def get(self, name: str) -> str:
     43     """ Returns the current value of a widget with the given name.
     44 
     45     :param name: Name of the argument to be accessed
   (...)
     80         ```
     81     """
---> 82     return self._notebookArguments.getArgument(name, self._entry_point.getCurrentBindings())

File /databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py:1362, in JavaMember.__call__(self, *args)
   1356 command = proto.CALL_COMMAND_NAME +\
   1357     self.command_header +\
   1358     args_command +\
   1359     proto.END_COMMAND_PART
   1361 answer = self.gateway_client.send_command(command)
-

In [0]:
bronze_table = spark.table(f"{config.catalog.catalog_name}.{config.catalog.source_schema}.{table}")

In [0]:
def transform_table(*,bronze_table:pyspark.sql.connect.dataframe.DataFrame, rate_type:str, update_frequency:str ) -> pyspark.sql.connect.dataframe.DataFrame:

    transformed_table = bronze_table.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))\
    .withColumnRenamed("code", "currency_code")\
        .withColumn("price_in_PLN_raw",  col("price").cast(DecimalType(10,4)))\
                .withColumn("currency_code", upper(trim(col("currency_code"))))\
                    .withColumn("ingest_date", current_date())\
                        .withColumn("rate_type", lit(rate_type))\
                            .withColumn("update_frequency", lit(update_frequency))\
                                .drop("price")

  
    valid_transformed_table = transformed_table.filter(
        (col("date").isNotNull()) &
        (col("currency_code").isNotNull()) &
        (col("price_in_PLN_raw").isNotNull()) &
        (col("price_in_PLN_raw") > 0) &
        (length(col("currency_code")) == 3)
    ).dropDuplicates(["date", "currency_code"])

    return valid_transformed_table

In [0]:
valid_table = transform_table(bronze_table=bronze_table, rate_type=rate_type, update_frequency=update_frequency)
display(valid_table.head(10))

date,currency_code,price_in_PLN_raw,ingest_date,rate_type,update_frequency
2025-10-21,THB,0.111400000000000000,2025-11-20,A,DAILY
2025-10-21,USD,3.647900000000000000,2025-11-20,A,DAILY
2025-10-21,AUD,2.365600000000000000,2025-11-20,A,DAILY
2025-10-21,HKD,0.469500000000000000,2025-11-20,A,DAILY
2025-10-21,CAD,2.594400000000000000,2025-11-20,A,DAILY


In [0]:
valid_table.write.format("delta").mode("overwrite").saveAsTable(f"{config.catalog.catalog_name}.{config.catalog.silver_schema}.{table}")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7139859878850388>, line 1
----> 1 valid_table.write.format("delta").mode("overwrite").saveAsTable(f"{config.catalog.catalog_name}.{config.catalog.silver_schema}.table_xxxx")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1556, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1554     req.user_context.user_id